# Modèle de langage et génération de séquence — Transformeurs

Dans ces travaux pratiques, nous allons étudier les différentes manières existantes pour décoder du texte étant donné un modèle entraîné. Cette étape est nécessaire pour beaucoup de tâches de traitement du langage, dont la traduction, le question/réponse et la génération.

Nous allons prendre l'exemple de la génération de texte : elle passait obligatoirement par des textes à trou plus ou moins sophistiqués jusqu'à très récemment. L'arrivée de modèles puissants entraînés sur de grandes quantités de données a changé la donne : il est maintenant possible d'utiliser des méthodes neuronales.

Dans ce notebook, nous allons voir comment générer du texte à partir d'un modèle entraîné, en partant de la méthode la plus simple pour progresser jusqu'aux méthodes état de l'art. Cela vous permettra d'asseoir solidement votre compréhension des mécanismes liés aux réseaux de neurones appliqués au texte : l'utilisation de l'activation softmax en particulier, qui est au cœur des principales architectures NLP état de l'art.

Un [excellent article](https://huggingface.co/blog/how-to-generate) de blog sur le sujet de Hugging Face peut servir de ressource complémentaire à ce notebook.

## Installation

La librairie [`transformers`](https://github.com/huggingface/transformers) développée par la startup franco-américaine [Hugging Face](https://huggingface.co/) met à disposition un grand nombre de modèles de la famille des transformeurs.

Elle s'installe facilement avec `pip` :

In [ ]:
!pip install transformers

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 48.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 14.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 73.3 MB/s eta 0:00:00


## Choix de la langue

Nous allons voir comment générer du texte en anglais mais aussi en français, en chargeant des modèles préentraînés sur des corpus différents. La cellule suivante permet de changer la langue utilisée pour le reste du TP.

In [ ]:
#lang = "fr"
lang = "en"

## Import d'un modèle entraîné

En plus de mettre à disposition le code source de nombreuses architectures de transformeurs, la librairie `transformers` permet de récupérer de [nombreux modèles entraînés](https://huggingface.co/transformers/pretrained_models.html).

Pour ce TP, nous allons utiliser l'architecture GPT2, derrière le [modèle qui a surpris la communauté NLP](https://openai.com/blog/better-language-models/) par la qualité de ses générations (et qui a depuis vu son successeur, GPT3, atteindre [des résultats encore plus bluffants](https://github.com/elyase/awesome-gpt3)).

Pour cela il suffit d'utiliser la méthode `from_pretrained` sur la classe de l'architecture du modèle qui nous intéresse, avec comme argument le nom de l'entrainement que l'on souhaite récupérer. On peut récupérer le tokeniseur lié au modèle de la même manière.

In [ ]:
import functools
import typing

import numpy
import tensorflow
import transformers
import tqdm.notebook


pretraining_name = "antoiloui/belgpt2" if lang == "fr" else "gpt2"

model = transformers.TFGPT2LMHeadModel.from_pretrained(pretraining_name)
tokenizer = transformers.GPT2Tokenizer.from_pretrained(pretraining_name)

All model checkpoint layers were used when initializing TFGPT2LMHeadModel.

All the layers of TFGPT2LMHeadModel were initialized from the model checkpoint at gpt2.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


## Utilisation du GPU

Par défaut, TensorFlow utilise les GPUs disponibles. Vérifions si Colab nous en a mis à disposition :

In [ ]:
print(tensorflow.config.experimental.list_physical_devices("GPU"))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Inspection du modèle

Après la récupération d'un modèle pré-entraîné, il est toujours bon de vérifier son architecture et ses caractéristiques.

Pour cela, dans la librairie `transformers`, on peut vérifier sa config :

- *Combien le modèle a-t-il de couches de transformeur ?*
- *Quels sont ses mécanismes de régularisation ?*
- *De stabilisation de l'apprentissage ?*
- *Nous aurons besoin plus tard de l'index du symbole spécial utilisé pendant l'entraînement pour délimiter les début et fin de texte. Quel est-il ?*

In [ ]:
print(model.config)

GPT2Config {
  "_name_or_path": "gpt2",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.29.1",
  "use_cache": true,
  "vocab_size": 50257
}



*Votre réponse à compléter ici.*
-
-
-
-

### Solution

- Il possède 12 couches
- Son principal mécanisme de régularisation est le dropout
- Ses mécanismes de stabilisation de l'apprentissage visibles depuis l'architecture montrée ci-dessus sont la layer normalization et les connexions résiduelles.
- On peut lire sa valeur dans les clefs `bos_token_id` et `eos_token_id` de la config : `50256`.

## Le mécanisme de génération

Pour générer du texte depuis un modèle entraîné, on fonctionne de manière itérative : on génère les mots un par un, en commençant avec un texte vide ou une phrase que l'on souhaite compléter.

Pour générer un mot, on donne en entrée du réseau les mots générés jusque là(transformés en indices d'embedding). Le réseau produit alors un score par mot du vocabulaire. On choisit l'un de ces mots avant de boucler (concaténer ce mot à ce qui avait été généré puis donner le résultat en entrée au réseau pour générer le mot suivant).

Plus formellement, on essaye de maximiser $P(\text{Texte}| \text{Initial})$ avec la décomposition suivante :

$$P(\text{Texte}| \text{Initial}) = \Pi_{i=1}^{\text{Taille du texte}} P(\text{Mot}_i| \text{Mot}_{j<i}, \text{Initial})$$

Tout l'enjeu pendant du decoding est de trouver la valeur maximale du produit : c'est la meilleure proposition du modèle. Cette valeur est impossible à calculer exactement, on a donc recours à des heuristiques pour l'approximer.

## Tokenisation

Le tokeniseur récupéré en même temps que le modèle permet de transformer une phrase d'entrée en index d'embedding que le modèle peut comprendre.

Nous utiliserons dans la suite du TP les fonctions `encode` et `decode` suivantes qui transforment des chaînes de caractères en index de vocabulaire que le modèle comprend (ils sont les index de son module d'embedding).

In [ ]:
def encode(sentence: str) -> tensorflow.Tensor:
  tokens = tokenizer.encode(sentence,
                            add_special_tokens=False,
                            return_tensors="tf")
  bos = tensorflow.constant([[model.config.bos_token_id]], dtype="int64")
  return tensorflow.concat([bos, tensorflow.cast(tokens, "int64")], axis=1)


def decode(tokens: tensorflow.Tensor) -> str:
  if tokens.ndim == 2:
    tokens = tensorflow.squeeze(tokens)
  return tokenizer.decode(tokens[1:], clean_up_tokenization_spaces=True)

## Étude des sorties du modèle pré-entraîné

La première étape pour décoder des phrases depuis un modèle appris est de faire une passe forward du modèle pour récupérer ses prédictions pour le prochain mot. Étudions son comportement.

Dans Keras, pour calculer la passe forward d'un modèle, on utilise directement `model(input_tensor)` comme nous l'avons déjà vu.

*Étudiez l'output du model pré-entrainé ([documentation](https://huggingface.co/transformers/model_doc/gpt2.html#tfgpt2model)) en l'appliquant à diverses séquences, par exemple :*

- *«&nbsp;&nbsp;»*
- *«&nbsp;I think therefore I am&nbsp;»* ou *«&nbsp;Je pense donc je suis&nbsp;» en fonction de la langue du modèle*

*En consultant la documentation, notez l'argument `past` : il permet d'éviter de recalculer des valeurs calculées pendant l'étape de decoding précédente. Si on donne l'argument past au modèle, il ne faut pas repasser les anciennes valeurs.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
def describe_shapes(input_string: str,
                    output_tensor: tensorflow.Tensor,
                    output_past: typing.List[tensorflow.Tensor]
                   ) -> str:
  print("-" * 80)
  print(f"Taille de l'output pour l'input '{input_string}' : "
        f"{output_tensor.shape}")
  print(f"Taille du passé : {len(output_past)}")
  print(f"Type du passé : {type(output_past[0])}")


empty_input = ""
descartes_input = ("Je pense, donc je suis"
                   if lang == "fr"
                   else "I think, therefore I am")

empty_output, empty_past = model(encode(empty_input),
                                 return_dict=False)
descartes_output, descartes_past = model(encode(descartes_input),
                                         return_dict=False)

describe_shapes(empty_input, empty_output, empty_past)
describe_shapes(descartes_input, descartes_output, descartes_past)

--------------------------------------------------------------------------------
Taille de l'output pour l'input '' : (1, 1, 50257)
Taille du passé : 12
Type du passé : <class 'tensorflow.python.framework.ops.EagerTensor'>
--------------------------------------------------------------------------------
Taille de l'output pour l'input 'I think, therefore I am' : (1, 7, 50257)
Taille du passé : 12
Type du passé : <class 'tensorflow.python.framework.ops.EagerTensor'>


## Fonction `forward` adaptée aux besoins du décodage

Étant donné les observations faites dans la section précédente, nous utiliserons la fonction `forward` suivante, qui rend un output adapté au décodage : seule la dernière colonne nous intéresse (la prédiction du mot suivant). Cette fonction `forward` implémente cette sélection.

In [ ]:
def forward(tokens: tensorflow.Tensor,
            past: typing.Optional[typing.List[tensorflow.Tensor]] = None
           ) -> typing.Tuple[tensorflow.Tensor, typing.List[tensorflow.Tensor]]:
  logits, new_past = model(tokens, past=past, return_dict=False)
  return logits[:, -1, :], new_past

Notez bien que quand l'argument `past` est utilisé, il ne faut pas redonner les tokens qui ont déjà été calculés, seulement le nouveau token à décoder. Il faut donc utiliser au choix :

- `forward(all_tokens)`
- `forward(last_token, past)`

où `all_tokens` serait par exemple une séquence de 7 tokens de forme `(1, 7, 50257)` là où `last_token` serait de forme `(1, 1, 50257)`.

*Pourquoi est-il souvent utile de conserver des dimensions à `1` dans les réseaux de neurones ?*

*Votre réponse à compléter ici.*

### Réponse

Ces dimensions peuvent souvent être utilisées pour calculer en parallèle des résultats pour plusieurs exemples (batching).

## Boucle de décodage

Une boucle de décodage suit toujours le même schéma :

1. Encodage de tout ce qui a été produit jusqu'ici (entrées et sorties précédentes)
2. Production de l'indice du mot suivant le plus probable
3. Répétition de 1. et 2. jusqu'à ce qu'un critère d'arrêt soit satisfait (nous utiliserons le nombre de mots produits seulement dans ce TP)

L'étape 2 est l'étape où tout se joue. Nous allons pour l'instant coder tout le reste, afin de pouvoir nous concentrer sur l'étape 2 par la suite.

In [ ]:
def decoding_loop(step_function) -> str:

  @functools.wraps(step_function)
  def wrapper(prompt, length, *step_args, **step_kwargs):
    token_ids = encode(prompt)
    past = None
    decoded = [token_ids]
    for i in tqdm.notebook.trange(length, desc="Mots", leave=False):
      logits, past = forward(decoded[-1], past)
      index = step_function(logits, *step_args, **step_kwargs)
      decoded += [index[None, None]]
    decoded_tensor = tensorflow.concat(decoded, axis=1)
    return decode(decoded_tensor)

  return wrapper

## Décodage greedy

Le décodage greedy est la méthode la plus simple pour décoder du texte depuis un modèle appris.

À chaque étape du décodage (pour décoder chaque mot), il consiste à prendre le mot avec la plus forte probabilité.

Plus formellement, dans l'équation :

$$P(\text{Texte}|\text{Initial}) = \Pi_{i=0}^{\text{Taille du texte}} P(\text{Mot}_i| \text{Mot}_{j<i}, \text{Initial})$$

Le décodage greedy choisira $\text{Mot}_i$ pour maximer $P(\text{Mot}_i| \text{Mot}_{j<i}, \text{Initial})$. Or, il faut souvent choisir un mot moins probable pour ensuite atteindre des mots très probables.

L'exemple suivant est donné dans le blog de Hugging Face sur la génération de texte :

![Greedy decoding](https://huggingface.co/blog/assets/02_how-to-generate/greedy_search.png)

On peut y voir qu'après avoir décodé «&nbsp;The&nbsp;», on choisit «&nbsp;nice&nbsp;» parce ce que c'est le mot avec la probabilité maximale pour le modèle et que le decoding complet «&nbsp;The nice woman&nbsp;» a une probabilité inférieure (0.20) au decoding que l'on aurait obtenu en choisissant «&nbsp;dog&nbsp;» («&nbsp;The dog has&nbsp;», probabilité 0.36).

*Codez la fonction `greedy(logits: tensorflow.Tensor) -> tensorflow.Tensor` qui prend en entrée l'output du modèle et sélectionne le mot à retourner (l'indice de la dernière dimension).*

*Elle utilisera [`tensorflow.math.argmax`](https://www.tensorflow.org/api_docs/python/tf/math/argmax). La fonction `decoding_loop` s'attend à un scalaire comme valeur de retour : un tenseur TensorFlow de forme `()`.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
@decoding_loop
def greedy(logits: tensorflow.Tensor) -> tensorflow.Tensor:
  return tensorflow.math.argmax(logits, axis=1)[0]


print(greedy("I went to the", 10))

Mots:   0%|          | 0/10 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/transformers/modeling_tf_utils.py:487: FutureWarning: The `past` argument is deprecated and will be removed in a future version, use `past_key_values` instead.
  warnings.warn(


I went to the store and bought a new pair of shoes. I


## Test sur des inputs variés

*Codez une méthode `test_decoding(function, *args, **kwargs) -> None` qui prend en entrée une méthode de decoding et ses arguments et montre les résultats de cette méthodes sur des inputs fixes. Testez la méthode `greedy` avec cette méthode. Que constatez-vous ?*

In [ ]:
if lang == "fr":
    inputs = ["Je suis allé au",
              "Je me sens",
              "Comment vas-tu",
              "Comment se fait-il que",
              "Ça va merci, tu devrais",
              "Donald Trump vient de tweeter :",
              "Le président Trump vient de démissionner !"]
else:
    inputs = ["I went to the",
              "I am feeling",
              "How do you",
              "How comes that",
              "I'm fine thank you, you should",
              "Donald Trump just tweeted :",
              "Le président Trump just quit !"]

In [ ]:
# Votre code ici

### Solution


In [ ]:
def test_decoding(function, *args, **kwargs) -> None:
  generations = []
  for i in tqdm.notebook.tqdm(inputs, desc="Prompts", leave=False):
    generations.append(function(i, *args, **kwargs))
  for prompt, generation in zip(inputs, generations):
    print("—" * 80)
    print(f"Prompt : {prompt}")
    print(f"Génération : {generation}")
  print("—" * 80)

test_decoding(greedy, 50)

Prompts:   0%|          | 0/7 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

————————————————————————————————————————————————————————————————————————————————
Prompt : I went to the
Génération : I went to the store and bought a new pair of shoes. I was very excited to see the new pair of shoes. I was very excited to see the new pair of shoes. I was very excited to see the new pair of shoes. I was very excited to
————————————————————————————————————————————————————————————————————————————————
Prompt : I am feeling
Génération : I am feeling a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit of a bit
————————————————————————————————————————————————————————————————————————————————
Prompt : How do you
Génération : How do you know if you're a good candidate for a job?

The answer is simple: you're not.

You're not a good candidate for a job because you're not a good candidate for a job.

You're not a
————————————————————————————————————————————————————————————————————————————————

On constate que les réponses «&nbsp;bouclent&nbsp;» assez rapidement. C'est une faille très classique du décodage greedy.

## Sampling

Comme nous venons de le constater, les résultats du décodage greedy posent de nombreux problèmes. Ils bouclent rapidement et sont souvent très génériques. L'objet des méthodes à suivre est de palier ces déficiences.

La première approche que nous allons voir consiste à échantilloner à partir des résultats du modèle plutôt que de toujours choisir le résultat le plus probable.

*Reprenez la forme de la fonction `greedy` mais au lieu de choisir l'élément maximal de l'output, calculez une distribution de probabilités à partir du modèle (passée au log pour la stabilité numérique), avec la fonction d'activation [`tensorflow.nn.log_softmax`](https://www.tensorflow.org/api_docs/python/tf/nn/log_softmax), et utilisez la méthode [`tensorflow.random.categorical`](https://www.tensorflow.org/api_docs/python/tf/random/categorical) pour échantilloner depuis cette distribution. Le résultat de cet échantillonage sera le prochain mot généré.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
@decoding_loop
def sampling(logits: tensorflow.Tensor) -> tensorflow.Tensor:
  softmaxed = tensorflow.nn.log_softmax(logits)
  return tensorflow.random.categorical(softmaxed, 1)[0][0]

test_decoding(sampling, 50)

Prompts:   0%|          | 0/7 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

————————————————————————————————————————————————————————————————————————————————
Prompt : I went to the
Génération : I went to the top state post of everywhere that is-less the minister, plebiscite, or respondent of a parliamentary suit and file not to harass me. NOIR came to town and forced me to dine with many others I gathered among his ten thousand
————————————————————————————————————————————————————————————————————————————————
Prompt : I am feeling
Génération : I am feeling really well, right now has been me other floods have felt like me to all the people of event out there but it isn't a miracle, just an illustration whatever people are calling it. Nor is it if it is hard to see in SW3
————————————————————————————————————————————————————————————————————————————————
Prompt : How do you
Génération : How do you catch sexton bats when there's no batting robot to patrol your field? English Clay, for example, gets caught with a safety device strapped to his eye.

Open a silencer

Indi

## Top-k sampling

Une variation de l'échantillonage consiste à n'échantilloner que depuis les `k` résultats maximaux, pour éviter de générer un mot trop improbable (même si cela n'arrive que rarement).

*Repartez de la fonction `sampling` pour coder la fonction `k_sampling(logits: tensorflow.Tensor, k: int) -> tensorflow.Tensor` qui implémente cette amélioration. Testez la avec la fonction `test_decoding`.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
@decoding_loop
def k_sampling(logits: tensorflow.Tensor, k: int) -> tensorflow.Tensor:
  topk = tensorflow.math.top_k(logits, k=k)
  softmaxed = tensorflow.nn.log_softmax(topk.values)
  sampled = tensorflow.random.categorical(softmaxed, 1)[0][0]
  return tensorflow.cast(topk.indices[0, sampled], "int64")

test_decoding(k_sampling, 50, k=20)

Prompts:   0%|          | 0/7 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

Mots:   0%|          | 0/50 [00:00<?, ?it/s]

————————————————————————————————————————————————————————————————————————————————
Prompt : I went to the
Génération : I went to the hospital today to talk with Dr. Thomas P. Hockley. We will have two more weeks of interviews later on Tuesday before we go to the hospital for a second visit. We will be able to discuss the various issues in the treatment center.
————————————————————————————————————————————————————————————————————————————————
Prompt : I am feeling
Génération : I am feeling very excited to see a release coming next year. I had no idea when it would release, but now that I can see it, I think I will be playing it.


The game is a cooperative cooperative action game where you play as an
————————————————————————————————————————————————————————————————————————————————
Prompt : How do you
Génération : How do you know when you're out in the wild?

Well, that's where I say "the wild" comes in…

Wild animals

Wild animals are animals that are not human beings (that's the name of wil

## Top-p sampling

Une autre amélioration de l'échantillonage consiste à ne considérer que les `n` premiers outputs, dont la probabilité sommée dépasse `p`, et pas les suivants.

*Repartez de la fonction `k_sampling` et ajoutez au top-k sampling le top-p sampling, dans la fonction `k_p_sampling(logits: tensorflow.Tensor, k: int, p: float) -> tensorflow.Tensor`. Testez la avec la fonction `test_decoding`.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
@decoding_loop
def k_p_sampling(logits: tensorflow.Tensor, k: int, p: float
                ) -> tensorflow.Tensor:
    topk = tensorflow.math.top_k(logits, k=k)
    softmaxed = tensorflow.nn.softmax(topk.values)
    current_proba_sum = 0
    j = 0
    while current_proba_sum < p:
      current_proba_sum += softmaxed[0, j]
      j += 1
    sampled = tensorflow.random.categorical(
        tensorflow.math.log(softmaxed[:, :j]), 1)[0][0]
    return tensorflow.cast(topk.indices[0, sampled], "int64")

test_decoding(k_p_sampling, length=100, k=20, p=0.85)

Prompts:   0%|          | 0/7 [00:00<?, ?it/s]

Mots:   0%|          | 0/100 [00:00<?, ?it/s]

Mots:   0%|          | 0/100 [00:00<?, ?it/s]

Mots:   0%|          | 0/100 [00:00<?, ?it/s]

Mots:   0%|          | 0/100 [00:00<?, ?it/s]

Mots:   0%|          | 0/100 [00:00<?, ?it/s]

Mots:   0%|          | 0/100 [00:00<?, ?it/s]

Mots:   0%|          | 0/100 [00:00<?, ?it/s]

————————————————————————————————————————————————————————————————————————————————
Prompt : I went to the
Génération : I went to the hospital today and my heart went out to the patient and he had no symptoms. He has a very difficult disease and we need a doctor to help him. The doctors told me he had no pulse. I went back and took him to my doctor. I was told I was in a very good place, and my heart was beating well. It's a very important thing to be in the right place at the right time.

"This is my first case of the type of heart attack I
————————————————————————————————————————————————————————————————————————————————
Prompt : I am feeling
Génération : I am feeling like I am in love with a woman, I want to have sex with her. It is so simple. I want to be with her and I want her to know how I feel. I want to feel like she is my wife, my husband, and she loves me too. I am going to go out on a date and I am going to make love to her. I'm going to go out on a date and she is going to be v

## Utilisation des fonctions de `transformers`

Dans le futur, si vous avez besoin de générer du texte, vous pourrez utiliser la fonction `generate` des modèles de la librarie `transformers`. Voici un exemple :

In [ ]:
inputs = ["Kim jong Un resign after a headache", "Donald Trump just tweeted :"]

In [ ]:
def transformers_generate(prompt: str,
                          length: int,
                          temperature: float,
                          top_k: int,
                          top_p: float) -> str:
  token_ids = encode(prompt)
  generated = model.generate(input_ids=tensorflow.cast(token_ids, "int32"),
                             max_length=len(token_ids) + length,
                             temperature=temperature,
                             top_k=top_k,
                             top_p=top_p,
                             do_sample=True,
                             num_return_sequences=1)
  return decode(tensorflow.cast(generated, "int64"))

test_decoding(transformers_generate, length=1000, temperature=1, top_k=30, top_p=0.95)

Prompts:   0%|          | 0/2 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


————————————————————————————————————————————————————————————————————————————————
Prompt : Kim jong Un resign after a headache
Génération : Kim jong Un resign after a headache-inducing four-hour interview on "The Oprah Winfrey Show" at the White House on Monday, as she seeks to change the course of her career and focus on personal growth.

"As a result, I am going to be more productive," Kim said as he spoke to reporters. "I'm going to be more of a leader."

The 33-year-old, who had been diagnosed with Parkinson's disease in 2011, went on to say she has not only become a mother, but the first female president in US history to be diagnosed with the disorder.

She went on to call herself a "personhood feminist" who believes that "women are born that way."

While her appearance was overshadowed by the fact that the former US senator from Alaska, who is believed to be on her way to become the first female president, was due to appear on the "Tonight Show" show, her candid comments about her